In [ ]:
import os
import numpy as np, pandas as pd
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.linear_model import Ridge
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

SEED = 42
paths = ["/kaggle/input/competitions/gdgoc-cye-ai-2026", "/kaggle/input/gdgoc-cye-ai-2026", ".", "/content"]
DATA = next((p for p in paths if os.path.exists(p + "/train.csv")), None)
if DATA is None:
    from google.colab import files
    files.upload()
    DATA = "."
OUT = "/kaggle/working" if os.path.exists("/kaggle/working") else "."

train = pd.read_csv(DATA + "/train.csv")
test = pd.read_csv(DATA + "/test.csv")

cols = ["dc_voltage", "dc_current", "irradiance", "panel_temp"]
med = train[cols].median()
train[cols] = train[cols].fillna(med)
test[cols] = test[cols].fillna(med)

classes = sorted(train.label.unique())
y = train.label.map({c: i for i, c in enumerate(classes)}).values
print(train.shape, test.shape)
print(train.label.value_counts())

In [ ]:
def xi(d): return np.c_[d.irradiance, d.panel_temp]
def xv(d): return np.c_[d.panel_temp, np.log(d.irradiance.clip(lower=1))]
def poly(): return make_pipeline(PolynomialFeatures(2), StandardScaler(), Ridge(alpha=1e-2))

def fit_baseline(df):
    h = df[(df.label == "normal") & (df.irradiance > 100)]
    return poly().fit(xi(h), h.dc_current), poly().fit(xv(h), h.dc_voltage)

def expected(model, d):
    mi, mv = model
    ev = mv.predict(xv(d))
    return np.clip(mi.predict(xi(d)), 0.1, None), np.where(np.abs(ev) < 1, 1, ev)

def expected_crossfit(df):
    ei, ev = np.zeros(len(df)), np.zeros(len(df))
    for a, b in GroupKFold(min(5, df.day.nunique())).split(df, groups=df.day):
        ei[b], ev[b] = expected(fit_baseline(df.iloc[a]), df.iloc[b])
    return ei, ev

def features(d, ei, ev):
    g = d.irradiance.clip(lower=1)
    i, v = d.dc_current, d.dc_voltage
    f = pd.DataFrame({
        "temp": d.panel_temp, "log_g": np.log(g),
        "i_over_g": i / g, "p_over_g": i * v / g, "v_over_i": v / i.clip(lower=0.1),
        "i_res": i - ei, "i_ratio": i / ei, "v_res": v - ev, "v_ratio": v / ev,
        "p_ratio": i * v / (ei * ev), "r_series": (ev - v) / i.clip(lower=0.5),
    })
    return f.replace([np.inf, -np.inf], np.nan).fillna(0)

In [ ]:
def weights(yy, power=0.5):
    counts = np.bincount(yy, minlength=len(classes))
    return ((len(yy) / (len(classes) * np.maximum(counts, 1))) ** power)[yy]

def models():
    return [
        RandomForestClassifier(n_estimators=200, min_samples_leaf=10, max_features=0.5, n_jobs=-1, random_state=SEED),
        HistGradientBoostingClassifier(max_depth=4, learning_rate=0.08, max_iter=150, min_samples_leaf=40,
                                       l2_regularization=1.0, random_state=SEED),
    ]

def fit_predict(Xtr, ytr, Xte):
    probs = []
    for m in models():
        m.fit(Xtr, ytr, sample_weight=weights(ytr))
        p = np.zeros((len(Xte), len(classes)))
        p[:, m.classes_] = m.predict_proba(Xte)
        probs.append(p)
    return np.mean(probs, axis=0)

In [ ]:
oof = np.zeros((len(train), len(classes)))
for tr, va in GroupKFold(min(5, train.day.nunique())).split(train, y, train.day):
    dtr, dva = train.iloc[tr], train.iloc[va]
    Xtr = features(dtr, *expected_crossfit(dtr))
    Xva = features(dva, *expected(fit_baseline(dtr), dva))
    oof[va] = fit_predict(Xtr, y[tr], Xva)

pred = oof.argmax(1)
print("macro F1:", round(f1_score(y, pred, average="macro"), 4))
print(classification_report(y, pred, target_names=classes, digits=3))

In [ ]:
X = features(train, *expected_crossfit(train))
Xt = features(test, *expected(fit_baseline(train), test))
pred = fit_predict(X, y, Xt).argmax(1)

sub = pd.DataFrame({"id": test.id, "label": [classes[i] for i in pred]})
sub.to_csv(OUT + "/submission.csv", index=False)
print(sub.label.value_counts(normalize=True).round(3))